In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("FlightDelayPredictionBigData") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.network.timeout", "800s") \
    .config("spark.executor.heartbeatInterval", "100s") \
    .config("spark.rdd.compress", "true") \
    .config("spark.memory.offHeap.enabled", "true") \
    .config("spark.memory.offHeap.size", "2g") \
    .getOrCreate()

HDFS_PATH = "hdfs://localhost:9000/data/"

In [2]:
airlines = spark.read.csv(HDFS_PATH + "airlines.csv", header=True, inferSchema=True, sep = ",")
airlines.show(2)

+---------+--------------------+
|IATA_CODE|             AIRLINE|
+---------+--------------------+
|       UA|United Air Lines ...|
|       AA|American Airlines...|
+---------+--------------------+
only showing top 2 rows


In [3]:
airports = spark.read.csv(HDFS_PATH + "airports.csv", header=True, inferSchema=True, sep = ",")
airports.show(2)

+---------+--------------------+---------+-----+-------+--------+---------+
|IATA_CODE|             AIRPORT|     CITY|STATE|COUNTRY|LATITUDE|LONGITUDE|
+---------+--------------------+---------+-----+-------+--------+---------+
|      ABE|Lehigh Valley Int...|Allentown|   PA|    USA|40.65236| -75.4404|
|      ABI|Abilene Regional ...|  Abilene|   TX|    USA|32.41132| -99.6819|
+---------+--------------------+---------+-----+-------+--------+---------+
only showing top 2 rows


In [4]:
flights = spark.read.csv(HDFS_PATH + "flights.csv", header=True, inferSchema=True, sep = ",")
flights.show(2)

+----+-----+---+-----------+-------+-------------+-----------+--------------+-------------------+-------------------+--------------+---------------+--------+----------+--------------+------------+--------+--------+---------+-------+-----------------+------------+-------------+--------+---------+-------------------+----------------+--------------+-------------+-------------------+-------------+
|YEAR|MONTH|DAY|DAY_OF_WEEK|AIRLINE|FLIGHT_NUMBER|TAIL_NUMBER|ORIGIN_AIRPORT|DESTINATION_AIRPORT|SCHEDULED_DEPARTURE|DEPARTURE_TIME|DEPARTURE_DELAY|TAXI_OUT|WHEELS_OFF|SCHEDULED_TIME|ELAPSED_TIME|AIR_TIME|DISTANCE|WHEELS_ON|TAXI_IN|SCHEDULED_ARRIVAL|ARRIVAL_TIME|ARRIVAL_DELAY|DIVERTED|CANCELLED|CANCELLATION_REASON|AIR_SYSTEM_DELAY|SECURITY_DELAY|AIRLINE_DELAY|LATE_AIRCRAFT_DELAY|WEATHER_DELAY|
+----+-----+---+-----------+-------+-------------+-----------+--------------+-------------------+-------------------+--------------+---------------+--------+----------+--------------+------------+--------+-

**FORMATING TIME TYPE FOR SCHEDULED_DEPARTURE, DEPARTURE_TIME, SCHEDULED_ARRIVAL, ARRIVAL_TIME**

# Example: 830 --> 08:30

In [5]:
from pyspark.sql.functions import col, when, lpad, concat, lit, substring

TIME_COLS = ['SCHEDULED_DEPARTURE', 'DEPARTURE_TIME', 'SCHEDULED_ARRIVAL', 'ARRIVAL_TIME']

for tcol in TIME_COLS:
    time_string_col = lpad(
        when(col(tcol) == 2400, 0)
        .otherwise(col(tcol))
        .cast("int").cast("string"),
        4, "0"
    )

    flights = flights.withColumn(
        tcol,
        concat(substring(time_string_col, 1, 2), lit(":"), substring(time_string_col, 3, 2))
    )


# Creating label --> Delay > 15 minutes --> 1 (Delay), otherwise 0 (In time)

In [6]:
from pyspark.sql import functions as F

flights = flights.filter(F.col("CANCELLED") == 0)

flights = flights.withColumn(
    "label",
    F.when(F.col("ARRIVAL_DELAY") > 15, 1).otherwise(0)
)

# Example 08:30 --> 8*60 + 30

In [7]:
flights = flights.withColumn(
    "DEP_TIME_MINUTES",
    F.split(F.col("DEPARTURE_TIME"), ":")[0].cast("int") * 60 +
    F.split(F.col("DEPARTURE_TIME"), ":")[1].cast("int")
)

In [8]:
flights.show(2)

+----+-----+---+-----------+-------+-------------+-----------+--------------+-------------------+-------------------+--------------+---------------+--------+----------+--------------+------------+--------+--------+---------+-------+-----------------+------------+-------------+--------+---------+-------------------+----------------+--------------+-------------+-------------------+-------------+-----+----------------+
|YEAR|MONTH|DAY|DAY_OF_WEEK|AIRLINE|FLIGHT_NUMBER|TAIL_NUMBER|ORIGIN_AIRPORT|DESTINATION_AIRPORT|SCHEDULED_DEPARTURE|DEPARTURE_TIME|DEPARTURE_DELAY|TAXI_OUT|WHEELS_OFF|SCHEDULED_TIME|ELAPSED_TIME|AIR_TIME|DISTANCE|WHEELS_ON|TAXI_IN|SCHEDULED_ARRIVAL|ARRIVAL_TIME|ARRIVAL_DELAY|DIVERTED|CANCELLED|CANCELLATION_REASON|AIR_SYSTEM_DELAY|SECURITY_DELAY|AIRLINE_DELAY|LATE_AIRCRAFT_DELAY|WEATHER_DELAY|label|DEP_TIME_MINUTES|
+----+-----+---+-----------+-------+-------------+-----------+--------------+-------------------+-------------------+--------------+---------------+--------+---

# Pipeline String, OneHot for categorical cols --> assembling vector --> model (Logistic Regression)

In [12]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator

categorical_cols = ["MONTH", "DAY_OF_WEEK", "AIRLINE", "ORIGIN_AIRPORT", "DESTINATION_AIRPORT"]
numeric_cols = ["DEP_TIME_MINUTES", "DISTANCE", "DEPARTURE_DELAY"]

stages = []
for col in categorical_cols:
    indexer = StringIndexer(inputCol=col, outputCol=col + "_Index", handleInvalid="keep")
    encoder = OneHotEncoder(inputCol=col + "_Index", outputCol=col + "_Vec")
    stages += [indexer, encoder]

assembler_inputs = [col + "_Vec" for col in categorical_cols] + numeric_cols
assembler = VectorAssembler(inputCols=assembler_inputs, outputCol="features")
stages.append(assembler)

lr = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    weightCol="classWeight"
    # maxIter=20,
    # regParam=0.1,
    # elasticNetParam=0.0
)
stages.append(lr)

pipeline = Pipeline(stages=stages)

# Splitting train vs test --> creating classWeight --> predicting

In [14]:
train_data, test_data = flights.randomSplit([0.8, 0.2], seed=42)

total_count = train_data.count()
delay_count = train_data.filter(F.col("label") == 1.0).count()
on_time_count = total_count - delay_count

weight_for_delay = total_count / (2.0 * delay_count)
weight_for_ontime = total_count / (2.0 * on_time_count)

train_data_weighted = train_data.withColumn(
    "classWeight",
    F.when(F.col("label") == 1.0, weight_for_delay).otherwise(weight_for_ontime)
)

model = pipeline.fit(train_data_weighted)

predictions = model.transform(test_data)

In [21]:
model_path = "model/flight_logistic_model"
model.write().overwrite().save(model_path)

# Showing the results

In [18]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

print("=== RESULTS ===")

roc_evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)
roc_auc = roc_evaluator.evaluate(predictions)
print(f"1. ROC-AUC Score: {roc_auc:.4f}")

acc_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="accuracy"
)
accuracy = acc_evaluator.evaluate(predictions)
print(f"2. Accuracy:      {accuracy:.4f}")

prec_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="weightedPrecision", metricLabel=1.0
)
precision = prec_evaluator.evaluate(predictions)
print(f"3. Precision:     {precision:.4f}")

rec_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="weightedRecall", metricLabel=1.0
)
recall = rec_evaluator.evaluate(predictions)
print(f"4. Recall:        {recall:.4f}")

f1_evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="f1", metricLabel=1.0
)
f1 = f1_evaluator.evaluate(predictions)
print(f"5. F1-Score:      {f1:.4f}")

=== RESULTS ===
1. ROC-AUC Score: 0.9357
2. Accuracy:      0.9073
3. Precision:     0.9154
4. Recall:        0.9073
5. F1-Score:      0.9102


In [19]:
from sklearn.metrics import classification_report
y_compare = predictions.select("label", "prediction").toPandas()

print("=== SKLEARN CLASSIFICATION REPORT ===")
print(classification_report(y_compare["label"], y_compare["prediction"], target_names=["On Time", "Delay"]))

=== SKLEARN CLASSIFICATION REPORT ===
              precision    recall  f1-score   support

     On Time       0.96      0.92      0.94    940667
       Delay       0.70      0.83      0.76    204304

    accuracy                           0.91   1144971
   macro avg       0.83      0.88      0.85   1144971
weighted avg       0.92      0.91      0.91   1144971



# Extracting and showing the importance of features

In [20]:
import pandas as pd
import numpy as np

lr_model = model.stages[-1]
coefficients = lr_model.coefficients.toArray()

features_metadata = predictions.schema["features"].metadata["ml_attr"]["attrs"]

all_features = []
for attr_type, attr_list in features_metadata.items():
    for attr in attr_list:
        all_features.append((attr["idx"], attr["name"]))

all_features.sort(key=lambda x: x[0])
feature_names = [x[1] for x in all_features]

feature_imp_df = pd.DataFrame({
    'Feature_Name': feature_names,
    'Coefficient': coefficients,
    'Absolute_Importance': np.abs(coefficients)
})

feature_imp_df = feature_imp_df.sort_values(by='Absolute_Importance', ascending=False).reset_index(drop=True)

print("\n=== THE MOST INFLUENTIAL FEATURES (LOGISTIC REGRESSION) ===")
print(feature_imp_df.head(15))


=== THE MOST INFLUENTIAL FEATURES (LOGISTIC REGRESSION) ===
                     Feature_Name  Coefficient  Absolute_Importance
0   DESTINATION_AIRPORT_Vec_13459   -10.560567            10.560567
1   DESTINATION_AIRPORT_Vec_11503   -10.085016            10.085016
2   DESTINATION_AIRPORT_Vec_13541    -9.866501             9.866501
3   DESTINATION_AIRPORT_Vec_12016    -9.082964             9.082964
4   DESTINATION_AIRPORT_Vec_13127    -9.074145             9.074145
5        ORIGIN_AIRPORT_Vec_10268    -8.990471             8.990471
6        ORIGIN_AIRPORT_Vec_13502    -8.609344             8.609344
7        ORIGIN_AIRPORT_Vec_11503    -8.378484             8.378484
8        ORIGIN_AIRPORT_Vec_14150    -6.021088             6.021088
9   DESTINATION_AIRPORT_Vec_15041    -5.897227             5.897227
10  DESTINATION_AIRPORT_Vec_11315    -4.304716             4.304716
11       ORIGIN_AIRPORT_Vec_12888    -4.297446             4.297446
12       ORIGIN_AIRPORT_Vec_10581    -3.893033         